# P7: RNN bilan matnlarni tasniflash
**Mavzu:** matnni ketma-ketlikka o'girish, `nn.Embedding`+`nn.RNN`+`Linear`, o'qitish, F1 baho
**Kun:** 8-kun amaliyoti — 26-iyun 2026, 09:30-10:50 (80 daqiqa)
**Juftlashgan ma'ruza:** L7 — Rekurrent neyron tarmoqlar (`d07_rnn.pdf`)
**Kapstone modul:** m07 `RNNClassifier`
**Ma'lumot:** uz_news_mini (online, 5000 sample). Offline: `d03_checkpoints/uz_sentiment_mini.txt` (ijobiy/salbiy).

## Bugungi maqsadlar (ma'ruza [C] dan)
1. RNN bir qadamini ($h_t=\tanh(W_h h_{t-1}+W_x x_t)$) qo'lda hisoblaymiz.
2. Matnni tokenlarga, ketma-ketlikka o'girib, padding qilamiz.
3. `nn.Embedding`+`nn.RNN`+`Linear` klassifikatorini o'qitamiz.
4. Test'da F1 baholaymiz va kapstone m07 moduliga ulaymiz.

## Vaqt taqsimoti
| Bo'lim | Vaqt | Mazmun |
|---|---|---|
| §1 Muhit | 4 daq | importlar, seedlar, OFFLINE_FALLBACK, HAS_TORCH |
| §2 Yaxlit natija | 8 daq | tayyor RNN — F1 va bashorat demo |
| §3 Periferiya (PRIMM) | 17 daq | m01 → ketma-ketlik + padding; DataLoader/collate |
| §4 Yadro | 35 daq | RNN qadami (qulflangan), model qurish/o'qitish, bashorat |
| §5 Loyihaga ulash | 13 daq | m07 moduli, import test, git |
| §6 Tadqiqot + yakun | 7 daq | RNN vs N-gram, vanishing gradient |

## 1. Muhit tekshiruvi
Seedlar, offline bayroq, m01 yo'li va `torch` bayrog'i. RNN demo CPU da ishlaydi.

In [ ]:
# Kaggle: torch oldindan o'rnatilgan (GPU shart emas — kichik RNN CPU da ishlaydi)
import os, sys, random, math, tempfile
import numpy as np

random.seed(42)
np.random.seed(42)

OFFLINE_FALLBACK = True          # bundled sentiment mini bilan uchdan-uchgacha ishlaydi

# torch ixtiyoriy: bor bo'lsa Kaggle yo'li (nn.RNN), yo'q bo'lsa toza-numpy RNN
try:
    import torch
    torch.manual_seed(42)
    HAS_TORCH = True
    print("torch mavjud:", torch.__version__)
except ImportError:
    HAS_TORCH = False
    print("torch yo'q — toza-numpy RNN ishlatiladi (offline rejim).")

MODULES = os.path.join("..", "..", "capstone", "modules")
if not os.path.isdir(MODULES):
    MODULES = os.path.join("capstone", "modules")
sys.path.insert(0, MODULES)

from m01_text_preprocessor import TextPreprocessor
print("numpy:", np.__version__, "| OFFLINE_FALLBACK =", OFFLINE_FALLBACK)


## 2. Yaxlit natija (avval manzil)
Quyida tayyor RNN klassifikatori sentiment ma'lumotida o'qitiladi va yangi sharhlarni
\texttt{ijobiy}/\texttt{salbiy} deb tasniflaydi. Tafsilotlarni keyingi bo'limlarda ochamiz.

In [ ]:
from m07_rnn_classifier import RNNClassifier
from sklearn.metrics import f1_score

SENT = os.path.join("..", "..", "course", "practices", "d03_checkpoints", "uz_sentiment_mini.txt")
if not os.path.exists(SENT):
    SENT = os.path.join("course", "practices", "d03_checkpoints", "uz_sentiment_mini.txt")

def yukla_sentiment(path):
    """reyting<TAB>matn -> (texts, labels); {4,5}->ijobiy, {1,2}->salbiy, 3 tashlanadi."""
    texts, labels = [], []
    for line in open(path, encoding="utf-8"):
        if not line.strip():
            continue
        rating, text = line.rstrip("\n").split("\t", 1)
        r = int(rating)
        if r >= 4:   texts.append(text); labels.append("ijobiy")
        elif r <= 2: texts.append(text); labels.append("salbiy")
    return texts, labels

texts, labels = yukla_sentiment(SENT)
print("Sentiment misollar:", len(texts), "| ijobiy:", labels.count("ijobiy"), "salbiy:", labels.count("salbiy"))

clf_demo = RNNClassifier()
clf_demo.fit(texts, labels, epochs=15, hidden_size=64, lr=0.01)
preds = [clf_demo.predict(t) for t in texts]
f1 = f1_score([l == "ijobiy" for l in labels], [p == "ijobiy" for p in preds])
print("Train F1:", round(f1, 3))
print("Bashorat:", clf_demo.predict("juda zor mahsulot, rahmat!"))


## 3. Tayyor kod bloki — matnni ketma-ketlikka o'girish (PRIMM)
Bu bo'lim periferiya. RNN kirishi --- token \textbf{indekslari} ketma-ketligi. Matnni m01
bilan tokenlab, lug'at quramiz va padding bilan bir xil uzunlikka keltiramiz.

**Bashorat qiling:** quyidagi katak ikkita qisqa matnni indekslarga o'girgach, kalta matn
qanday "to'ldiriladi" (padding)? PAD indeksi nechta?

In [ ]:
# PERIFERIYA — to'liq beriladi. m01 -> token indekslari + padding.
pre = TextPreprocessor()

def lugat_qur(texts):
    vocab = sorted({t for tx in texts for t in pre.preprocess(tx)})
    return {w: i + 1 for i, w in enumerate(vocab)}    # 0 = PAD

def ketma_ketlikka(text, w2i):
    return [w2i[t] for t in pre.preprocess(text) if t in w2i] or [0]

def padding(seqs):
    T = max(len(s) for s in seqs)
    return [s + [0] * (T - len(s)) for s in seqs]      # 0 bilan to'ldirish

w2i = lugat_qur(texts)
demo_seqs = [ketma_ketlikka(t, w2i) for t in texts[:2]]
print("Lug'at hajmi:", len(w2i))
print("Ikki ketma-ketlik (padsiz):", demo_seqs)
print("Padding bilan:", padding(demo_seqs))


**Tekshiring:**
1. Nega PAD uchun aynan `0` indeks ajratildi? (Maslahat: `nn.Embedding(..., padding_idx=0)`.)
2. Padding qisqa matnni uzun matn uzunligiga keltiradi — bu nima uchun kerak?
3. m01 stop-so'zlarni oladi; bu sentimentga yordam beradimi yoki ziyon?

**O'zgartiring:** o'z sharhingizni `ketma_ketlikka(...)` ga bering va indekslarini ko'ring.

In [ ]:
# PERIFERIYA — Kaggle'da PyTorch DataLoader + collate_fn (offline'da oddiy batcher).
if HAS_TORCH:
    from torch.utils.data import DataLoader, TensorDataset
    def batcher(seqs, ys, bs=16):
        X = torch.tensor(padding(seqs), dtype=torch.long)
        y = torch.tensor(ys, dtype=torch.long)
        return DataLoader(TensorDataset(X, y), batch_size=bs, shuffle=True)
    print("DataLoader tayyor (Kaggle yo'li).")
else:
    def batcher(seqs, ys, bs=16):
        return [(padding(seqs), ys)]          # offline: bitta batch
    print("Oddiy batcher (offline yo'li).")


### Checkpoint A
Agar orqada qolsangiz — quyidagi katak ma'lumotni qaytadan yuklaydi (toza holat).

In [ ]:
if OFFLINE_FALLBACK or "texts" not in dir():
    texts, labels = yukla_sentiment(SENT)
print("Checkpoint: ma'lumot tayyor —", len(texts), "misol")


## 4. Asosiy mavzu — RNN ni qadam-baqadam (so'nuvchi tayanch)
**Namuna → Birgalikda → Mustaqil.** Avval ma'ruzaning qo'lda misolini takrorlaymiz.

### 4A. Namuna (men qilaman): RNN bir qadami
Ma'ruza L7 [I2]-slaydida: $h_0=[0,0]$, $W_h=W_x=I$, $x_1=[1,0]$. RNN qadami ---
$h_t=\tanh(W_h h_{t-1}+W_x x_t)$. Buni qo'lda (toza-numpy) hisoblaymiz.

In [ ]:
# RNN bir qadami (toza-numpy, torch shart emas) — L7 [I2]
Wh = np.eye(2)            # birlik matritsa I_2
Wx = np.eye(2)
h0 = np.zeros(2)
x1 = np.array([1.0, 0.0])

h1 = np.tanh(Wh @ h0 + Wx @ x1)
print("h1 =", h1)

assert np.allclose(h1, [0.762, 0.0], atol=1e-3)   # Ma'ruza L7 [I2]-slayd bilan solishtiring
print("To'g'ri! h1 = tanh([1,0]) = [0.762, 0.000]")


### 4B. Birgalikda (biz qilamiz): RNN klassifikatorini o'qitamiz
`RNNClassifier` ichida `nn.Embedding`+`nn.RNN`+`Linear` (Kaggle) yoki toza-numpy RNN
(offline). Modelni sentiment ma'lumotida o'qitamiz. CPU uchun kichik o'lcham, kam epoch.

In [ ]:
clf = RNNClassifier()
# Modelni o'qiting: epochs=15, hidden_size=64, lr=0.01
clf.fit(texts, labels, epochs=15, hidden_size=64, lr=0.01)


In [ ]:
assert len(clf._w2i) > 0, "Lug'at bo'sh — fit() chaqirilganini tekshiring."
assert clf.predict("juda zor mahsulot rahmat") in {"ijobiy", "salbiy"}, \
    "predict() 'ijobiy' yoki 'salbiy' qaytarishi kerak."
print("Ajoyib! Model o'qitildi — lug'at:", len(clf._w2i), "so'z; yorliqlar:", clf._labels)


### 4C. Birgalikda (biz qilamiz): ehtimollik bashorati
`predict_proba` har sinf uchun ehtimol beradi (softmax). Yig'indisi 1 ga teng bo'lishi kerak.

In [ ]:
pr = clf.predict_proba("juda zor mahsulot, rahmat!")
print(pr)


In [ ]:
assert set(pr.keys()) == {"ijobiy", "salbiy"}, "Kalitlar {ijobiy, salbiy} bo'lishi kerak."
assert abs(sum(pr.values()) - 1.0) < 1e-5, "Ehtimolliklar yig'indisi 1 ga teng bo'lishi kerak."
print("To'g'ri! predict_proba {ijobiy, salbiy}, yig'indi 1.")


### 4D. Mustaqil (siz qilasiz): baholang va saqlang
Train ma'lumotida F1 (yoki aniqlik) hisoblang, modelni saqlang va qayta yuklang.
Yuklangan model bir xil bashorat berishini tekshiring. Tayanch yo'q.

In [ ]:
from sklearn.metrics import f1_score

preds = [clf.predict(t) for t in texts]
acc = sum(p == y for p, y in zip(preds, labels)) / len(labels)
f1 = f1_score([l == "ijobiy" for l in labels], [p == "ijobiy" for p in preds])
print("Train aniqlik:", round(acc, 2), "| F1:", round(f1, 3))

path = os.path.join(tempfile.gettempdir(), "m07_rnn.pkl")
clf.save(path)
clf2 = RNNClassifier(); clf2.load(path)

assert acc >= 0.6, "Model train'da kamida 0.6 aniqlikka yetishi kerak (o'qishni tekshiring)."
assert clf2.predict("juda zor mahsulot rahmat") == clf.predict("juda zor mahsulot rahmat"), \
    "Saqlab-yuklagandan keyin bashorat o'zgarmasligi kerak."
print("Mustaqil topshiriq bajarildi: model o'qidi, save/load mos.")


## 5. Loyihaga ulash — m07 `RNNClassifier`
Bugungi ish allaqachon `capstone/modules/m07_rnn_classifier.py` da (shartnoma
`capstone/contracts.py`). U torch-ixtiyoriy: Kaggle'da `nn.RNN`, offline'da toza-numpy.
Quyida import qilib, shartnomaga mosligini tekshiramiz.

In [ ]:
from m07_rnn_classifier import RNNClassifier

m = RNNClassifier()
for metod in ["fit", "predict", "predict_proba", "save", "load"]:
    assert callable(getattr(m, metod, None)), f"m07.{metod} mavjud emas!"
print("m07 RNNClassifier shartnomaga mos: fit / predict / predict_proba / save / load")


Kapstone repozitoriyingizga saqlang:
```bash
git add capstone/modules/m07_rnn_classifier.py
git commit -m "P7: m07 RNNClassifier — RNN sentiment tasnif"
git push
```

## 6. Tadqiqot savoli + yakun
**Mini-tajriba:** quyidagi katak uzun sharh oxiriga ijobiy so'z qo'shib, RNN bashoratini
kuzatadi. RNN gap \textbf{oxiridagi} so'zga qanchalik sezgir? (SOV — kesim oxirida.)

In [ ]:
qisqa = "mahsulot keldi"
uzun  = "mahsulot keldi lekin sifati past edi va umuman yoqmadi afsus"
print("qisqa  ->", clf.predict(qisqa), clf.predict_proba(qisqa))
print("uzun   ->", clf.predict(uzun),  clf.predict_proba(uzun))
print("Eslatma: uzun gapda oxirgi so'zlar h_T ga ko'proq ta'sir qiladi (vanishing gradient).")


**Savol (o'ylab ko'ring):** RNN gap oxiridagi so'zlarni boshidagidan yaxshiroq "eslaydi"
(vanishing gradient). O'zbekcha SOV (kesim oxirida) bunga yordam beradimi yoki ziyonmi?
Uzoq bog'liqlik uchun qanday model kerak? (Maslahat: L8 — LSTM/GRU.)

**Chiqish chiptasi:** _Bugun eng tushunarsiz qolgan narsa nima?_